In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("MySparkApp") \
    .getOrCreate()


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/08/31 07:10:52 WARN Utils: Your hostname, MacBook-Pro-6.local, resolves to a loopback address: 127.0.0.1; using 100.100.165.223 instead (on interface en0)
26/08/31 07:10:52 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/08/31 07:10:53 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [16]:
data = [
    {"userId": 1, "paymentAmount": 100.0, "date": "2025-01-01"},
    {"userId": 2, "paymentAmount": 150.5, "date": "2025-01-02"},
    {"userId": 3, "paymentAmount": 200.75, "date": "2025-01-03"},
    {"userId": 2, "paymentAmount": 50.25, "date": "2025-01-04"},
    {"userId": 1, "paymentAmount": 80.0, "date": "2025-01-05"}
    
]

df = spark.createDataFrame(data)
df.show()


+----------+-------------+------+
|      date|paymentAmount|userId|
+----------+-------------+------+
|2025-01-01|        100.0|     1|
|2025-01-02|        150.5|     2|
|2025-01-03|       200.75|     3|
|2025-01-04|        50.25|     2|
|2025-01-05|         80.0|     1|
+----------+-------------+------+



In [3]:
df.count()

5

In [2]:
listings = spark.read.csv("/Users/kamari/Documents/project_info/air_bnb/listings.csv.gz",                 
    header=True,
    inferSchema=True,
    sep= "," , 
    quote = '"' , 
    escape='"' , 
    multiLine=True,
    mode="PERMISSIVE"
    )

listings.show(5)



26/08/31 07:10:57 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+-------+--------------------+--------------+------------+-----------+--------------------+--------------------+---------------------+--------------------+--------+--------------------+-------------------+--------------------+---------+----------+------------------------+-------------------------+------------------------+-------------------------+-------------+--------------------+------------------+------------------+--------------------+-----------------+------------------+--------------------+------------------+-------------------+-------------------------+------------------+--------------------+----------------------+-------------+----------------------+----------------------------+--------+---------+--------------------+---------------+------------+---------+--------------+--------+----+--------------------+-------+------------------------+-------------------------+-----------------------+---------------------------+--------------------+--------------+--------------+--------------

In [5]:
for field in listings.schema:
    print(field)

StructField('id', LongType(), True)
StructField('listing_url', StringType(), True)
StructField('scrape_id', LongType(), True)
StructField('last_scraped', DateType(), True)
StructField('source', StringType(), True)
StructField('name', StringType(), True)
StructField('description', StringType(), True)
StructField('neighborhood_overview', StringType(), True)
StructField('picture_url', StringType(), True)
StructField('host_id', LongType(), True)
StructField('host_url', StringType(), True)
StructField('host_profile_id', LongType(), True)
StructField('host_profile_url', StringType(), True)
StructField('host_name', StringType(), True)
StructField('host_since', StringType(), True)
StructField('hosts_time_as_user_years', IntegerType(), True)
StructField('hosts_time_as_user_months', IntegerType(), True)
StructField('hosts_time_as_host_years', IntegerType(), True)
StructField('hosts_time_as_host_months', IntegerType(), True)
StructField('host_location', StringType(), True)
StructField('host_about

In [6]:
neighborhoods = listings.select(listings.neighbourhood_cleansed)
neighborhoods.show()

+----------------------+
|neighbourhood_cleansed|
+----------------------+
|            THIRD WARD|
|            SIXTH WARD|
|           SECOND WARD|
|            SIXTH WARD|
|           SECOND WARD|
|       FOURTEENTH WARD|
|        FIFTEENTH WARD|
|         ELEVENTH WARD|
|            SIXTH WARD|
|        FIFTEENTH WARD|
|            TENTH WARD|
|        FIFTEENTH WARD|
|            NINTH WARD|
|       FOURTEENTH WARD|
|            FIFTH WARD|
|            TENTH WARD|
|        FIFTEENTH WARD|
|            NINTH WARD|
|       FOURTEENTH WARD|
|        FIFTEENTH WARD|
+----------------------+
only showing top 20 rows


In [7]:
review_locations = listings.select(listings.review_scores_location)
review_locations.show()

+----------------------+
|review_scores_location|
+----------------------+
|                  3.22|
|                  4.82|
|                  4.75|
|                   4.8|
|                  4.77|
|                  4.93|
|                  4.94|
|                  4.52|
|                  4.72|
|                  4.81|
|                  4.67|
|                  4.85|
|                  4.82|
|                  4.92|
|                  4.67|
|                   4.7|
|                  4.97|
|                  4.95|
|                  4.88|
|                  4.86|
+----------------------+
only showing top 20 rows


In [8]:
listings\
    .select(listings.review_scores_location) \
    .show()

+----------------------+
|review_scores_location|
+----------------------+
|                  3.22|
|                  4.82|
|                  4.75|
|                   4.8|
|                  4.77|
|                  4.93|
|                  4.94|
|                  4.52|
|                  4.72|
|                  4.81|
|                  4.67|
|                  4.85|
|                  4.82|
|                  4.92|
|                  4.67|
|                   4.7|
|                  4.97|
|                  4.95|
|                  4.88|
|                  4.86|
+----------------------+
only showing top 20 rows


In [9]:
high_score_listings = listings \
    .filter(listings.review_scores_location > 4.5) \
    .select("id","price","name", "review_scores_location")
high_score_listings.show()

+--------+-------+--------------------+----------------------+
|      id|  price|                name|review_scores_location|
+--------+-------+--------------------+----------------------+
| 3820211|$211.00|Restored Precinct...|                  4.82|
| 5651579| $97.00|Large studio apt ...|                  4.75|
| 6623339|$160.00|Center Sq. Loft i...|                   4.8|
| 9501054| $86.00|Spacious suite wi...|                  4.77|
|10768745| $58.00|Alb hospital area...|                  4.93|
|11253948|$277.31|/Fire Place Bunga...|                  4.94|
|11639446| $62.87|$55twin($30 forei...|                  4.52|
|12799126| $64.00|Private Room in t...|                  4.72|
|13083497|$313.00|Pristine Cape Cod...|                  4.81|
|14316232| $51.89|Unique Modern Roo...|                  4.67|
|14989805|$394.00|/Miller Colonial\...|                  4.85|
|15580397| $68.50|Albany Medical/Do...|                  4.82|
|16531782| $92.00|On a little park ...|                

In [10]:
high_score_listings.dropna().show(20,truncate=False)

+--------+-------+--------------------------------------------------+----------------------+
|id      |price  |name                                              |review_scores_location|
+--------+-------+--------------------------------------------------+----------------------+
|3820211 |$211.00|Restored Precinct in Center Sq. w/Parking         |4.82                  |
|5651579 |$97.00 |Large studio apt  by Capital Center & ESP@        |4.75                  |
|6623339 |$160.00|Center Sq. Loft in Converted Precinct w/ Parking  |4.8                   |
|9501054 |$86.00 |Spacious suite with full bath by Capital  Center  |4.77                  |
|10768745|$58.00 |Alb hospital area studio bath wifi. (Red)         |4.93                  |
|11253948|$277.31|/Fire Place Bungalow\ 1917 SUNY Eagle 6Beds 2Baths|4.94                  |
|11639446|$62.87 |$55twin($30 foreign student)FreeBF Noa/c no smoke |4.52                  |
|12799126|$64.00 |Private Room in the Hearth of the Albany          |4

In [3]:
from pyspark.sql.functions import regexp_replace

price_num_df = listings \
    .withColumn("price_num", regexp_replace("price", "[$,]", "").cast("float"))
    
price_num_df.schema['price_num']

StructField('price_num', FloatType(), True)

In [12]:
price_num_df.filter((price_num_df.price_num < 100) & (price_num_df.review_scores_location > 4.5)) \
    .select("name", "price", "review_scores_location") \
    .show(truncate=False)

+--------------------------------------------------+------+----------------------+
|name                                              |price |review_scores_location|
+--------------------------------------------------+------+----------------------+
|Large studio apt  by Capital Center & ESP@        |$97.00|4.75                  |
|Spacious suite with full bath by Capital  Center  |$86.00|4.77                  |
|Alb hospital area studio bath wifi. (Red)         |$58.00|4.93                  |
|$55twin($30 foreign student)FreeBF Noa/c no smoke |$62.87|4.52                  |
|Private Room in the Hearth of the Albany          |$64.00|4.72                  |
|Unique Modern Room, Perfect Location Albany       |$51.89|4.67                  |
|Albany Medical/Downtown Albany/Colleges/ ROOM H   |$68.50|4.82                  |
|On a little park in Albany pine hills. (Blue)     |$92.00|4.92                  |
|$53($25 foreign student)Twin, noa/cno Smoke freeBF|$60.67|4.67                  |
|Coz

In [13]:
listings \
    .select(listings.property_type, listings.room_type) \
    .distinct() \
    .show(truncate = False)

+---------------------------------+---------------+
|property_type                    |room_type      |
+---------------------------------+---------------+
|Entire serviced apartment        |Entire home/apt|
|Private room in villa            |Private room   |
|Room in hotel                    |Hotel room     |
|Private room in guest suite      |Private room   |
|Entire rental unit               |Entire home/apt|
|Private room in home             |Private room   |
|Entire loft                      |Entire home/apt|
|Private room in bed and breakfast|Private room   |
|Room in aparthotel               |Entire home/apt|
|Entire home                      |Entire home/apt|
|Entire condo                     |Entire home/apt|
|Entire guesthouse                |Entire home/apt|
|Entire place                     |Entire home/apt|
|Entire cottage                   |Entire home/apt|
|Entire guest suite               |Entire home/apt|
|Entire vacation home             |Entire home/apt|
|Room in hot

Exercises

In [14]:
# 1. Get a non-null picture URL for any property ("picture_url" field)
# Select any non-null picture URL
non_null_pictures = listings.select(listings.picture_url).isNotNull()
non_null_pictures.show(1)

#same but different
listings.filter(
    listings.picture_url.isNotNull()
) \
.select("picture_url")   \
.limit(1) \
.show(truncate= False)


PySparkAttributeError: [ATTRIBUTE_NOT_SUPPORTED] Attribute `isNotNull` is not supported.

In [ ]:
# 2. Get number of properties that get more than 10 reviews per month

listings.filter(listings.reviews_per_month > 10).count()


2

In [ ]:
# 3. Get properties that have more bathrooms than bedrooms


listings \
.filter(listings.bathrooms > listings.bedrooms) \
.select("name", "bathrooms",  "bedrooms") \
.show()





+--------------------+---------+--------+
|                name|bathrooms|bedrooms|
+--------------------+---------+--------+
|Triplex oasis w:p...|      1.5|       1|
|Modern Duplex - H...|      1.5|       1|
|'The Morris Home'...|      2.0|       1|
|'The Morris Home'...|      2.0|       1|
|‘The Morris Home’...|      2.0|       1|
|Large Bedroom in ...|      1.5|       1|
|‘The Morris Home’...|      2.0|       1|
|Large Bedroom in ...|      1.5|       1|
|Large room with w...|      1.5|       1|
|    AMC Cozy Bedroom|      2.0|       1|
+--------------------+---------+--------+



26/08/26 13:38:48 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:70)
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:44)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:359)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRefByURI(RpcEnv.scala:102)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRef(RpcEnv.scala:110)
	at org.apache.spark.util.RpcUtils$.makeDriverRef(RpcUtils.scala:34)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.driverEndpoint$lzycompute(BlockManagerMasterEndpoint.scala:132)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.org$apache$spark$storage$BlockManagerMasterEndpoint$$driverEndpoint(BlockManagerMasterEndpoint.scala:131)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.isExecutorAlive$lzycompute$1(Blo

In [ ]:
# 4. Get properties where the price is greater than 500. Collect the result as a Python list
# Remember to convert a price into a number first!

from pyspark.sql.functions import regexp_replace

price_num_df = listings \
    .withColumn("price_num", regexp_replace("price", "[$,]", "").cast("float"))


more_than_5k = price_num_df.filter(
    (price_num_df.price_num  > 500)
    ) \
     .select("name","price") \
     .collect()  

more_than_5k


[Row(name='/zBig Blue Ranch\\ 1962 SUNY Eagle Hill 4Beds 2Bath', price='$845.75'),
 Row(name='/ New Giant Victorian \\ 7beds 6 baths + 2x 85" TVs', price='$1,040.50'),
 Row(name='+ Perfect place to make memories with loved ones +', price='$819.00'),
 Row(name='/ Sauna Garden Getaway\\ 1943 SUNY Eagle Hill', price='$700.50'),
 Row(name='The Historic Jesse Buel Farmhouse', price='$1,701.00'),
 Row(name='🪴 Historic & Homey 1 Bdrm Apt @ Downtown Albany', price='$1,267.00'),
 Row(name='Historic Washington Park Inn | Full Mansion | 7 BR', price='$840.00'),
 Row(name='Modern Cottage-Perfect for Families!', price='$587.00'),
 Row(name='5BR/4BA Retreat | Fire Pit + BBQ', price='$625.00'),
 Row(name='ABBA House Retreat: Centrally Located Grand Manor', price='$1,265.00'),
 Row(name='Massive 5BD/4BA 3-Story Duplex @ Downtown Albany', price='$531.00'),
 Row(name='Downtown ALB • Hot Tub • Game Room • Free Parking', price='$540.00'),
 Row(name='Hot Tub Patio + 2 Ensuites | Walk to Capitol & MVP', pri

In [ ]:
#4.1 Find all listings with more than 3 reviews per month.

listings \
.filter(listings.reviews_per_month > 3) \
.select('name', 'reviews_per_month') \
.show()

+--------------------+-----------------+
|                name|reviews_per_month|
+--------------------+-----------------+
|Spacious suite wi...|             3.71|
|Alb hospital area...|             7.18|
|On a little park ...|              8.2|
|Quiet and Pretty ...|             10.2|
|Private Room in H...|             3.03|
|Historic Loft Sui...|             8.14|
|Cozy Garden 2-Bed...|             7.57|
|Historic Full Ame...|             5.53|
|Historic, Spaciou...|             5.79|
|Perfect Downtown ...|             3.97|
|     Comfy and quiet|             5.95|
|Stunning Bungalo-...|             3.16|
|Historic Queen St...|             5.57|
|Parkside Queen 1 ...|             4.61|
|Our Antique Bungalow|             4.29|
|The Loft Suite @ ...|             3.27|
|Downtown Albany 2...|             4.35|
|Beautiful Center ...|             3.75|
|Studio in Heart o...|              3.7|
|Chic-romantic, ce...|             4.65|
+--------------------+-----------------+
only showing top

In [ ]:
# 4.2 Find the top 5 most expensive listings.
from pyspark.sql.functions import regexp_replace

price_num_df = listings \
    .withColumn("price_num", regexp_replace("price", "[$,]", "").cast("float"))

price_num_df.sort(price_num_df.price_num.desc()) \
    .select('name', 'price') \
    .limit(5)   \
    .show()

+--------------------+---------+
|                name|    price|
+--------------------+---------+
|The Historic Jess...|$1,701.00|
|🪴 Historic & Hom...|$1,267.00|
|ABBA House Retrea...|$1,265.00|
|/ New Giant Victo...|$1,040.50|
|/zBig Blue Ranch\...|  $845.75|
+--------------------+---------+



In [ ]:
#4.3 Find all listings where the number of bathrooms is greater than the number of bedrooms.


listings \
.filter(listings.bathrooms > listings.bedrooms) \
.select("name", "bathrooms",  "bedrooms") \
.show()




+--------------------+---------+--------+
|                name|bathrooms|bedrooms|
+--------------------+---------+--------+
|Triplex oasis w:p...|      1.5|       1|
|Modern Duplex - H...|      1.5|       1|
|'The Morris Home'...|      2.0|       1|
|'The Morris Home'...|      2.0|       1|
|‘The Morris Home’...|      2.0|       1|
|Large Bedroom in ...|      1.5|       1|
|‘The Morris Home’...|      2.0|       1|
|Large Bedroom in ...|      1.5|       1|
|Large room with w...|      1.5|       1|
|    AMC Cozy Bedroom|      2.0|       1|
+--------------------+---------+--------+



In [ ]:
#4.4 Find the number of listings for each room type.
listings \
.groupBy(listings.room_type) \
.count() \
.show()


+---------------+-----+
|      room_type|count|
+---------------+-----+
|    Shared room|    1|
|     Hotel room|    9|
|Entire home/apt|  345|
|   Private room|  135|
+---------------+-----+



In [ ]:
#4.5 Find listings that have: at least 3 bedrooms at least 2 bathrooms accommodate at least 6 people

listings \
.filter((listings.bathrooms >= 2) & (listings.bedrooms >=3 )& (listings.accommodates >= 6 )) \
.show()



+-------------------+--------------------+--------------+------------+-----------+--------------------+--------------------+---------------------+--------------------+---------+--------------------+-------------------+--------------------+-------------------+----------+------------------------+-------------------------+------------------------+-------------------------+--------------------+--------------------+------------------+------------------+--------------------+-----------------+------------------+--------------------+------------------+-------------------+-------------------------+------------------+--------------------+----------------------+-------------+----------------------+----------------------------+------------------+------------------+--------------------+---------------+------------+---------+--------------+--------+----+--------------------+---------+------------------------+-------------------------+-----------------------+---------------------------+--------------

In [42]:
# 5. Get a list of properties with the following characteristics:
# * price < 150
# * more than 20 reviews
# * review_scores_rating > 4.5
# Consider using the "&" operator


from pyspark.sql.functions import regexp_replace

price_num_df = listings \
    .withColumn("price_num", regexp_replace("price", "[$,]", "").cast("float"))

price_num_df \
.filter((price_num_df.price_num < 150) & (price_num_df.number_of_reviews > 20) & (listings.review_scores_rating > 4.5))\
.show()


+--------+--------------------+--------------+------------+-----------+--------------------+--------------------+---------------------+--------------------+---------+--------------------+-------------------+--------------------+---------+----------+------------------------+-------------------------+------------------------+-------------------------+----------------+--------------------+------------------+------------------+--------------------+-----------------+------------------+--------------------+------------------+-------------------+-------------------------+------------------+--------------------+----------------------+-------------+----------------------+----------------------------+--------+---------+--------------------+---------------+------------+---------+----------------+--------+----+--------------------+-------+------------------------+-------------------------+-----------------------+---------------------------+--------------------+--------------+--------------+-------

In [18]:
# 6. Get a list of properties with the following characteristics:
# * price < 150 OR more than one bathroom
# Use the "|" operator to implement the OR operator

price_num_df\
.filter((price_num_df.price_num < 150 ) | ( price_num_df.bathrooms > 1))\
.show()


+--------+--------------------+--------------+------------+-----------+--------------------+--------------------+---------------------+--------------------+---------+--------------------+-------------------+--------------------+-------------------+----------+------------------------+-------------------------+------------------------+-------------------------+----------------+--------------------+------------------+------------------+--------------------+-----------------+------------------+--------------------+------------------+-------------------+-------------------------+------------------+--------------------+----------------------+-------------+----------------------+----------------------------+---------+----------+--------------------+---------------+------------+---------+----------------+--------+----+--------------------+-------+------------------------+-------------------------+-----------------------+---------------------------+--------------------+--------------+----------

In [26]:
# 7. Get the highest listing price in this dataset
# Consider using the "max" function from "pyspark.sql.functions"

from pyspark.sql.functions import max

price_num_df\
.select(max(price_num_df.price_num)) \
.show()

+--------------+
|max(price_num)|
+--------------+
|        1701.0|
+--------------+



In [49]:
# 8. Get the name and a price of property with the highest price
# Try to use "collect" method to get the highest price first, and then use it in a "filter" call 
from pyspark.sql.functions import max

res = price_num_df\
.select (max(price_num_df.price_num).alias("max_price"))\
.collect() 

max_price = res[0]['max_price']

max_price


price_num_df\
.filter(price_num_df.price_num ==  max_price)\
.select('name', 'price')\
.show()


+--------------------+---------+
|                name|    price|
+--------------------+---------+
|The Historic Jess...|$1,701.00|
+--------------------+---------+



In [63]:
#8.1 — Lowest Priced Property Get the name and price of the property with the lowest price.
from pyspark.sql.functions import min

res = price_num_df.select(min(price_num_df.price_num).alias("min_price"))\
.collect()

min_price = res[0]["min_price"]

price_num_df \
.filter(price_num_df.price_num == min_price)\
.select('name','price')\
.show()

+--------------------+------+
|                name| price|
+--------------------+------+
|Quiet Room Near E...|$13.18|
+--------------------+------+



In [66]:
#8.2 Find the average number of reviews across all listings. Then find all properties whose number_of_reviews is greater than that average.
from pyspark.sql.functions import avg

res = listings.select(avg(listings.number_of_reviews).alias('avg_listings'))\
    .collect()
    
avg_list = res[0]["avg_listings"]

listings \
.filter(listings.number_of_reviews > avg_list) \
.select('name', 'number_of_reviews')\
.show()

+--------------------+-----------------+
|                name|number_of_reviews|
+--------------------+-----------------+
|Restored Precinct...|              317|
|Large studio apt ...|              406|
|Center Sq. Loft i...|              332|
|Spacious suite wi...|              469|
|Alb hospital area...|              906|
|/Fire Place Bunga...|              227|
|$55twin($30 forei...|              231|
|Pristine Cape Cod...|              200|
|/Miller Colonial\...|              143|
|On a little park ...|              604|
|$53($25 foreign s...|              132|
|Charming 1 Bedroo...|               92|
|5BedroomHome InAl...|              132|
|/Red Warhol Ranch...|              102|
|Garden Apartment,...|              132|
|     The Blair Suite|              140|
|NEW LISTING! Mode...|              191|
|Elegant Guest Sui...|              142|
|Quiet and Pretty ...|              996|
|Garden Apartment ...|              173|
+--------------------+-----------------+
only showing top

In [23]:
#8.3 Most Available Property Find the highest availability_365 value in the dataset.

from pyspark.sql.functions import max
res = listings.select(max(listings.availability_365).alias('highest_365'))\
.collect()

highest_365 = res[0]["highest_365"]
highest_365

listings\
.filter(listings.availability_365 == highest_365)\
.select('name','room_type',)\
.show()

+--------------------+---------------+
|                name|      room_type|
+--------------------+---------------+
|Sophisticated Ret...|Entire home/apt|
|               Homey|Entire home/apt|
|Center Sq Brownst...|Entire home/apt|
|Charming 1 Bdrm G...|Entire home/apt|
|🪴 Historic & Hom...|Entire home/apt|
|Beautiful 4 bedro...|Entire home/apt|
|Global Entry Unit...|Entire home/apt|
|Room close to AMe...|   Private room|
|Albany Cozy Stay ...|Entire home/apt|
|Lovely Albany roo...|Entire home/apt|
|Elegant 1BR in Hi...|Entire home/apt|
|Modern Albany Roo...|Entire home/apt|
|Cozy Albany Home ...|Entire home/apt|
|Relaxing 1BD Stay...|Entire home/apt|
|Central Albany Ho...|Entire home/apt|
|       Serenity Room|   Private room|
+--------------------+---------------+



In [81]:
# 9. Get the number of hosts in the dataset

listings\
.select(listings.host_name)\
.distinct()\
.count()

169

In [16]:
# 10. Get listings with a first review in 2024. Consider using the "year" function from "pyspark.sql.functions"
from pyspark.sql.functions import year

listings.filter(year(listings.first_review )== '2024')\
.select("name", 'first_review')\
.show()
                
     


+--------------------+------------+
|                name|first_review|
+--------------------+------------+
|      Moroccan Sands|  2024-10-28|
|    Cozy Albany Home|  2024-07-26|
|Your luxury home ...|  2024-02-18|
|Hudson 4 at The A...|  2024-09-02|
|The spacious room...|  2024-01-14|
|Garden level apar...|  2024-01-23|
|5BR/4BA Retreat |...|  2024-02-09|
|Elegant Albany Re...|  2024-02-17|
|*Huge, bright, pr...|  2024-07-31|
|2 bed room Apartment|  2024-02-11|
|Fully Furnished 2...|  2024-01-02|
|Bonjour, Bienvenu...|  2024-10-26|
|Massive 5BD/4BA 3...|  2024-02-23|
|Nighthawk room, p...|  2024-02-01|
|Lark St Oasis: Pa...|  2024-02-18|
|Newly renovated c...|  2024-11-24|
|Triplex oasis w:p...|  2024-02-29|
|Walkable Studio: ...|  2024-03-10|
|Homey, Quiet 2BR ...|  2024-04-14|
|Rowhouse 03 1 BDR...|  2024-03-10|
+--------------------+------------+
only showing top 20 rows


In [17]:
print(listings.columns)

['id', 'listing_url', 'scrape_id', 'last_scraped', 'source', 'name', 'description', 'neighborhood_overview', 'picture_url', 'host_id', 'host_url', 'host_profile_id', 'host_profile_url', 'host_name', 'host_since', 'hosts_time_as_user_years', 'hosts_time_as_user_months', 'hosts_time_as_host_years', 'hosts_time_as_host_months', 'host_location', 'host_about', 'host_response_time', 'host_response_rate', 'host_acceptance_rate', 'host_is_superhost', 'host_thumbnail_url', 'host_picture_url', 'host_neighbourhood', 'host_listings_count', 'host_total_listings_count', 'host_verifications', 'host_has_profile_pic', 'host_identity_verified', 'neighbourhood', 'neighbourhood_cleansed', 'neighbourhood_group_cleansed', 'latitude', 'longitude', 'property_type', 'room_type', 'accommodates', 'bathrooms', 'bathrooms_text', 'bedrooms', 'beds', 'amenities', 'price', 'price_quote_checkin_date', 'price_quote_checkout_date', 'price_quote_total_price', 'price_quote_price_per_night', 'price_quote_raw', 'minimum_nig

In [20]:
#10.1 — Long-Stay Listings. Find all listings that require guests to stay for at least 7 nights but no more than 30 nights.

listings\
.filter((listings.minimum_nights >= 7) & (listings.maximum_nights <= 30))\
.select('name', 'minimum_nights','maximum_nights' )\
.show()

+--------------------+--------------+--------------+
|                name|minimum_nights|maximum_nights|
+--------------------+--------------+--------------+
|Albany 3 Bedroom ...|             7|            27|
|A peaceful home a...|             7|            30|
+--------------------+--------------+--------------+



In [26]:
#10.2 — Recent Reviews Find all listings whose most recent review occurred in 2024 and that have received more than 25 reviews total.

listings\
.filter((listings.number_of_reviews > 25) & (year(listings.last_review)== 2024))\
.select('name', 'number_of_reviews','last_review')\
.show()

+--------------------+-----------------+-----------+
|                name|number_of_reviews|last_review|
+--------------------+-----------------+-----------+
|$55twin($30 forei...|              231| 2024-02-29|
|$53($25 foreign s...|              132| 2024-08-01|
|Cozy Modern Room ...|               56| 2024-11-02|
|/Sauna Ranch 1961...|               58| 2024-08-13|
|Sophisticated Ret...|               54| 2024-08-14|
|*The Blue House* ...|              131| 2024-10-28|
+--------------------+-----------------+-----------+



In [ ]:
listings\
.select('name','number_of_reviews')\
.sort(listings.number_of_reviews.desc())\
.limit(1)\
.show()

In [ ]:
#10.3 — Most Reviewed Listing. Find the property or properties with the highest total number of reviews in the dataset.
from pyspark.sql import functions

res = listings.select('name','number_of_reviews')\
.sort(listings.number_of_reviews.desc())\
.limit(1)\
.collect()

max_review_num = res[0]['number_of_reviews']
max_review_num

listings\
.filter(listings.number_of_reviews == max_review_num)\
.select('name','number_of_reviews')\
.show()



+--------------------+-----------------+
|                name|number_of_reviews|
+--------------------+-----------------+
|Quiet and Pretty ...|              996|
+--------------------+-----------------+



In [30]:
#10.4 — Average Rating by Room Type. Calculate the average review rating for each type of room and display the results from highest average rating to lowest.

from pyspark.sql import functions as f

listings\
.groupBy(listings.room_type)\
.agg(f.avg(listings.review_scores_rating).alias("avg_review_scores_rating"))\
.sort(f.desc('avg_review_scores_rating'))\
.show()



+---------------+------------------------+
|      room_type|avg_review_scores_rating|
+---------------+------------------------+
|   Private room|        4.79559633027523|
|Entire home/apt|       4.688714733542319|
|    Shared room|                    4.67|
|     Hotel room|                     1.0|
+---------------+------------------------+



In [39]:
listings.select(listings.neighbourhood_cleansed).show()

+----------------------+
|neighbourhood_cleansed|
+----------------------+
|            THIRD WARD|
|            SIXTH WARD|
|           SECOND WARD|
|            SIXTH WARD|
|           SECOND WARD|
|       FOURTEENTH WARD|
|        FIFTEENTH WARD|
|         ELEVENTH WARD|
|            SIXTH WARD|
|        FIFTEENTH WARD|
|            TENTH WARD|
|        FIFTEENTH WARD|
|            NINTH WARD|
|       FOURTEENTH WARD|
|            FIFTH WARD|
|            TENTH WARD|
|        FIFTEENTH WARD|
|            NINTH WARD|
|       FOURTEENTH WARD|
|        FIFTEENTH WARD|
+----------------------+
only showing top 20 rows


In [52]:
#10.5 Neighborhood Analysis. Calculate the average number of reviews per listing for each neighborhood. Keep only neighborhoods where the average is greater than 20, then display the results from highest average to lowest.
from pyspark.sql import functions as f

listings\
.groupBy(listings.neighbourhood_cleansed)\
.agg(f.avg(listings.number_of_reviews).alias("avg_number_of_reviews"))\
.filter(f.col("avg_number_of_reviews") > 20)\
.sort(f.desc("avg_number_of_reviews"))\
.show()



+----------------------+---------------------+
|neighbourhood_cleansed|avg_number_of_reviews|
+----------------------+---------------------+
|           SECOND WARD|    136.9512195121951|
|        FIFTEENTH WARD|                 94.0|
|       FOURTEENTH WARD|             91.03125|
|           EIGHTH WARD|    81.46153846153847|
|            SIXTH WARD|    70.05982905982906|
|           FOURTH WARD|    66.06666666666666|
|          SEVENTH WARD|   46.857142857142854|
|            NINTH WARD|   46.023809523809526|
|       THIRTEENTH WARD|   43.048780487804876|
|            FIRST WARD|                 37.0|
|            THIRD WARD|   31.488372093023255|
|         ELEVENTH WARD|                 30.5|
|            TENTH WARD|   27.267857142857142|
+----------------------+---------------------+



In [ ]:
#10.6 — Property Type Pricing. Calculate the average price for each property type. Keep only property types with an average price greater than $200, then display the results from highest average price to lowest.
from pyspark.sql import functions as f
from pyspark.sql.functions import regexp_replace

price_num_df = listings \
    .withColumn("price_num", regexp_replace("price", "[$,]", "").cast("float"))
    
price_num_df\
.groupBy(price_num_df.property_type)\
.agg(f.avg(price_num_df.price_num).alias("avg_property_type_price"))\
.filter(f.col("avg_property_type_price")>200)\
.sort(f.desc("avg_property_type_price"))\
.show()

+--------------------+-----------------------+
|       property_type|avg_property_type_price|
+--------------------+-----------------------+
|Shared room in be...|                  840.0|
|      Entire cottage|                  700.5|
|Entire serviced a...|                  538.5|
|   Entire guesthouse|                  431.0|
|         Entire home|      355.2735004425049|
|Entire vacation home|                  315.0|
|        Entire place|     277.30999755859375|
|    Entire townhouse|      240.7759979248047|
|       Room in hotel|     225.44444444444446|
|        Entire condo|     213.16666793823242|
+--------------------+-----------------------+



In [17]:
#10.7 — High-Availability Room Types. Calculate the average yearly availability for each room type. Keep only room types with an average availability greater than 150 days, then display the results from highest average availability to lowest.
from pyspark.sql import functions as f

listings\
.groupBy(listings.room_type)\
.agg(f.avg(listings.availability_365).alias('avg_availability'))\
.filter(f.col("avg_availability") > 150)\
.sort(f.col("avg_availability").desc())\
.show()

+---------------+------------------+
|      room_type|  avg_availability|
+---------------+------------------+
|     Hotel room| 288.1111111111111|
|Entire home/apt| 252.8608695652174|
|   Private room|215.16296296296295|
+---------------+------------------+



In [34]:
#10.8 — Most Common Property Type. Determine which property type has the most listings in the dataset. Display the property type and its total number of listings.

listings\
.groupBy(listings.property_type)\
.count()\
.sort(f.col('count').desc())\
.limit(1)\
.show()

+------------------+-----+
|     property_type|count|
+------------------+-----+
|Entire rental unit|  255|
+------------------+-----+

